In [ ]:
# ============================================================
# PROJECT 2
# INTELLIGENT MEETING TRANSCRIPTION & ACTION-ITEM EXTRACTOR
# ============================================================

!pip -q install flask

import threading
import time
import re

from flask import Flask, request, jsonify, render_template_string
from google.colab import output


# ============================================================
# NLP FUNCTIONS
# ============================================================

def extract_sentences(text):

    return [
        s.strip()
        for s in re.split(
            r"[.!?]+",
            text
        )
        if s.strip()
    ]


def extract_keywords(text):

    words = re.findall(
        r"\b[a-zA-Z]{4,}\b",
        text.lower()
    )

    stopwords = {
        "this","that","with",
        "from","have","will",
        "they","their","there",
        "about","which","would",
        "could","should","into",
        "need","also","then"
    }

    words = [
        w for w in words
        if w not in stopwords
    ]

    freq = {}

    for word in words:
        freq[word] = freq.get(
            word, 0
        ) + 1

    return sorted(
        freq,
        key=freq.get,
        reverse=True
    )[:8]


def extract_action_items(text):

    sentences = extract_sentences(text)

    action_words = [
        "will",
        "need to",
        "should",
        "must",
        "responsible",
        "assigned",
        "complete",
        "finish",
        "prepare",
        "submit",
        "send",
        "test",
        "develop",
        "create"
    ]

    actions = []

    for sentence in sentences:

        lower = sentence.lower()

        if any(
            word in lower
            for word in action_words
        ):

            actions.append(sentence)

    return actions


def create_summary(text):

    sentences = extract_sentences(text)

    if len(sentences) <= 3:
        return sentences

    keywords = extract_keywords(text)

    scored = []

    for sentence in sentences:

        score = sum(
            sentence.lower().count(k)
            for k in keywords
        )

        scored.append(
            (score, sentence)
        )

    scored.sort(
        reverse=True
    )

    return [
        sentence
        for score, sentence
        in scored[:3]
    ]


def analyze_meeting(text):

    if not text.strip():

        return {
            "error":
            "Please enter meeting content."
        }

    sentences = extract_sentences(text)

    words = text.split()

    return {

        "summary":
            create_summary(text),

        "actions":
            extract_action_items(text),

        "keywords":
            extract_keywords(text),

        "sentences":
            len(sentences),

        "words":
            len(words),

        "characters":
            len(text)
    }


# ============================================================
# FLASK
# ============================================================

app = Flask(__name__)


HTML = """

<!DOCTYPE html>

<html>

<head>

<meta name="viewport"
content="width=device-width,initial-scale=1">

<title>
Meeting Intelligence System
</title>

<style>

body {

    margin:0;

    padding:25px;

    font-family:Arial;

    background:
    linear-gradient(
        135deg,
        #172554,
        #0f766e
    );
}


.container {

    max-width:1000px;

    margin:auto;
}


h1 {

    color:white;

    text-align:center;
}


.subtitle {

    color:#dbeafe;

    text-align:center;
}


.card {

    background:white;

    padding:25px;

    margin-top:20px;

    border-radius:20px;
}


textarea {

    width:100%;

    height:180px;

    padding:15px;

    box-sizing:border-box;

    border-radius:12px;

    border:2px solid #ddd;

    font-size:16px;
}


button {

    padding:12px 20px;

    border:0;

    border-radius:10px;

    margin-top:12px;

    cursor:pointer;

    font-weight:bold;
}


.speak {

    background:#16a34a;

    color:white;
}


.analyze {

    background:#2563eb;

    color:white;
}


.clear {

    background:#dc2626;

    color:white;
}


.grid {

    display:grid;

    grid-template-columns:
    repeat(3,1fr);

    gap:15px;
}


.stat {

    background:#f3f4f6;

    padding:20px;

    border-radius:15px;

    text-align:center;
}


.value {

    font-size:25px;

    font-weight:bold;
}


.item {

    background:#eff6ff;

    padding:12px;

    border-radius:10px;

    margin:8px 0;
}


.keyword {

    display:inline-block;

    background:#dcfce7;

    color:#166534;

    padding:7px 12px;

    border-radius:20px;

    margin:4px;
}


</style>

</head>


<body>


<div class="container">


<h1>
🧠 Intelligent Meeting Assistant
</h1>

<p class="subtitle">
Speech Transcription • Summary • Action Items • Keywords
</p>


<div class="card">

<h2>
🎤 Meeting Transcript
</h2>


<textarea
id="meeting"
placeholder="Type your meeting discussion or use the microphone..."
></textarea>


<br>


<button
class="speak"
onclick="startSpeech()">

🎤 Record Speech

</button>


<button
class="analyze"
onclick="analyze()">

🔍 Analyze Meeting

</button>


<button
class="clear"
onclick="clearAll()">

🗑 Clear

</button>


</div>


<div class="card">

<h2>
📊 Meeting Statistics
</h2>


<div class="grid">


<div class="stat">

Words

<div class="value"
id="words">-</div>

</div>


<div class="stat">

Sentences

<div class="value"
id="sentences">-</div>

</div>


<div class="stat">

Characters

<div class="value"
id="characters">-</div>

</div>


</div>

</div>


<div class="card">

<h2>
📌 AI Summary
</h2>

<div id="summary">
-
</div>


<h2>
✅ Action Items
</h2>

<div id="actions">
-
</div>


<h2>
🔑 Important Topics
</h2>

<div id="keywords">
-
</div>

</div>


</div>


<script>


async function analyze(){

    const text =
        document.getElementById(
            "meeting"
        ).value;

    if(!text.trim()){

        alert(
            "Enter meeting content first."
        );

        return;
    }


    const response =
        await fetch(
            "/analyze",
            {

                method:"POST",

                headers:{
                    "Content-Type":
                    "application/json"
                },

                body:JSON.stringify({
                    text:text
                })

            }
        );


    const data =
        await response.json();


    if(data.error){

        alert(data.error);

        return;

    }


    document.getElementById(
        "words"
    ).innerText =
        data.words;


    document.getElementById(
        "sentences"
    ).innerText =
        data.sentences;


    document.getElementById(
        "characters"
    ).innerText =
        data.characters;


    // Summary

    const summary =
        document.getElementById(
            "summary"
        );

    summary.innerHTML="";


    data.summary.forEach(
        function(item){

            const div =
                document.createElement(
                    "div"
                );

            div.className="item";

            div.innerText="📌 "+item;

            summary.appendChild(div);

        }
    );


    // Actions

    const actions =
        document.getElementById(
            "actions"
        );

    actions.innerHTML="";


    if(data.actions.length===0){

        actions.innerHTML =
            "<p>No clear action items detected.</p>";

    }
    else{

        data.actions.forEach(
            function(item){

                const div =
                    document.createElement(
                        "div"
                    );

                div.className="item";

                div.innerText="✅ "+item;

                actions.appendChild(div);

            }
        );

    }


    // Keywords

    const keywords =
        document.getElementById(
            "keywords"
        );

    keywords.innerHTML="";


    data.keywords.forEach(
        function(item){

            const span =
                document.createElement(
                    "span"
                );

            span.className =
                "keyword";

            span.innerText =
                item;

            keywords.appendChild(
                span
            );

        }
    );

}


function startSpeech(){

    const SpeechRecognition =
        window.SpeechRecognition ||
        window.webkitSpeechRecognition;


    if(!SpeechRecognition){

        alert(
            "Please use Google Chrome."
        );

        return;

    }


    const recognition =
        new SpeechRecognition();


    recognition.lang="en-IN";


    recognition.continuous=true;


    recognition.interimResults=false;


    recognition.onstart=function(){

        alert(
            "🎤 Meeting recording started. Speak now."
        );

    };


    recognition.onresult=function(event){

        let result="";


        for(
            let i=0;
            i<event.results.length;
            i++
        ){

            result +=
                event.results[i][0].transcript
                + " ";

        }


        document.getElementById(
            "meeting"
        ).value=result;

    };


    recognition.onerror=function(event){

        alert(
            "Speech error: "
            + event.error
        );

    };


    recognition.start();

}


function clearAll(){

    location.reload();

}

</script>

</body>

</html>

"""


@app.route("/")
def home():

    return render_template_string(
        HTML
    )


@app.route(
    "/analyze",
    methods=["POST"]
)
def analyze():

    data = request.get_json()

    return jsonify(
        analyze_meeting(
            data.get("text","")
        )
    )


# ============================================================
# START SERVER
# ============================================================

def run_server():

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


threading.Thread(
    target=run_server,
    daemon=True
).start()


time.sleep(5)


url = output.eval_js(
    "google.colab.kernel.proxyPort(5000)"
)


print("\n======================================")
print("🚀 PROJECT 2 SERVER STARTED")
print("======================================")
print(url)
print("\nCLICK THE LINK ABOVE")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



🚀 PROJECT 2 SERVER STARTED
https://5000-m-s-kkb-usc1c2-eh8s7au7trng-c.us-central1-2.prod.colab.dev

CLICK THE LINK ABOVE
